# OvisOCR2 — image → HTML, comparable to VL 1.6 (Kaggle)

Test [`ATH-MaaS/OvisOCR2`](https://huggingface.co/ATH-MaaS/OvisOCR2): a compact ~0.8B document-parsing VLM (built on Qwen3.5-0.8B) that reads a page image and emits a single Markdown document with **HTML tables**, **LaTeX** formulas, and `<img>` tags for figures.

**Two halves:**
1. **On Kaggle (needs GPU):** run the model on the 100 images and save, per image, an HTML page + the raw Markdown + a per-image manifest (latency, chars, table count).
2. **Locally (last section):** score those outputs against the **VL 1.6** results we already have, using the *same* metric functions as `ocr_vl16_comparison.ipynb`, so OvisOCR2's numbers line up with run1–run4.

**Kaggle runtime:** Settings → Accelerator → GPU (T4/P100), Internet ON (model + MathJax download at runtime).

## 1. Install dependencies
The model card pins vLLM 0.22.1.

In [ ]:
%pip install -q "vllm==0.22.1" pillow markdown
# FlashInfer JIT-compiles + autotunes its kernels on T4 (compute 7.5) and hangs at
# model load. Removing it makes vLLM fall back to the TRITON_ATTN attention backend.
%pip uninstall -q -y flashinfer flashinfer-python 2>/dev/null || echo "flashinfer not installed"

## 2. Config
Point `IMAGE_DIR` at the same 100 images used for the VL 1.6 run (add them as a Kaggle Dataset; they mount under `/kaggle/input/...`).

In [ ]:
from pathlib import Path

MODEL_ID   = "ATH-MaaS/OvisOCR2"
IMAGE_DIR  = "/kaggle/input"                 # searched recursively for images
OUT_DIR    = Path("/kaggle/working/ovisocr2_html")
MD_DIR     = Path("/kaggle/working/ovisocr2_md")
MANIFEST   = Path("/kaggle/working/ovisocr2_per_image.csv")
EXTS       = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tiff", ".tif", ".jfif"}
MAX_IMAGES = None                            # set an int for a quick smoke test

# categories used across the experiments (prefix of the file name)
CATEGORIES = ("agenda", "form", "invoice", "report", "original")

def category_of(stem: str) -> str:
    for c in CATEGORIES:
        if stem.lower().startswith(c):
            return c
    return "other"

OUT_DIR.mkdir(parents=True, exist_ok=True)
MD_DIR.mkdir(parents=True, exist_ok=True)

image_paths = sorted(p for p in Path(IMAGE_DIR).rglob("*") if p.suffix.lower() in EXTS)
if MAX_IMAGES:
    image_paths = image_paths[:MAX_IMAGES]
print(f"Found {len(image_paths)} images")
for p in image_paths[:10]:
    print("  ", p.name, "->", category_of(p.stem))

## 3. The OvisOCR2 parser
Reference inference class from the model card (vLLM), verbatim. Loads the model, applies the fixed OCR prompt, returns Markdown per image.

In [ ]:
import os

# --- T4 (compute cap 7.5) fixes ---------------------------------------------
# FlashInfer JIT-compiles + autotunes its kernels on T4 and hangs at model load.
# It is uninstalled in the install cell so vLLM falls back to TRITON_ATTN for
# attention. Here we only disable the FlashInfer top-p/top-k SAMPLER (separate path).
# NOTE: VLLM_ATTENTION_BACKEND is NOT recognized by this vLLM 0.22.1 fork (it logs
# "Unknown vLLM environment variable" and is ignored) -> that's why we remove the
# flashinfer package rather than trying to force the backend via an env var.
os.environ.setdefault("VLLM_USE_FLASHINFER_SAMPLER", "0")

# --- HF token from Kaggle secret (faster, un-throttled weight download) ------
if not os.environ.get("HF_TOKEN"):
    try:
        from kaggle_secrets import UserSecretsClient
        os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
        print("HF_TOKEN loaded from Kaggle secrets.")
    except Exception as e:
        print("HF_TOKEN not loaded from Kaggle secrets (continuing unauthenticated):", e)

from PIL import Image
from vllm import LLM, SamplingParams


class OvisOCR2Parser:
    def __init__(self, model_name_or_path: str):
        self.model = LLM(
            model=model_name_or_path,
            tensor_parallel_size=1,
            gpu_memory_utilization=0.8,
            gdn_prefill_backend="triton",
            enforce_eager=True,   # skip torch.compile + CUDA-graph capture (the silent T4 stall)
        )

        prompt = '\nExtract all readable content from the image in natural human reading order and output the result as a single Markdown document. For charts or images, represent them using an HTML image tag: <' + 'img src="images/bbox_{left}_{top}_{right}_{bottom}.jpg" />, where left, top, right, bottom are bounding box coordinates scaled to [0, 1000). Format formulas as LaTeX. Format tables as HTML: <table>...</table>. Transcribe all other text as standard Markdown. Preserve the original text without translation or paraphrasing.'
        self.prompt = self.model.get_tokenizer().apply_chat_template(
            [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": prompt}]}],
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=False,
        )

        self.sampling_params = SamplingParams(
            max_tokens=16384,
            temperature=0.0,
        )

    def _clean_truncated_repeats(
        self,
        text: str,
        min_text_len: int = 8000,
        max_period: int = 200,
        min_period: int = 1,
        min_repeat_chars: int = 100,
        min_repeat_times: int = 5,
    ) -> str:
        n = len(text)
        if n < min_text_len:
            return text

        max_period = min(max_period, n - 1)
        for unit_len in range(min_period, max_period + 1):
            if text[n - 1] != text[n - 1 - unit_len]:
                continue

            match_len = 1
            idx = n - 2
            while idx >= unit_len and text[idx] == text[idx - unit_len]:
                match_len += 1
                idx -= 1

            total_len = match_len + unit_len
            repeat_times = total_len // unit_len
            tail_len = total_len % unit_len

            if repeat_times >= min_repeat_times and total_len >= min_repeat_chars:
                return text[: n - total_len + unit_len] + text[n - tail_len:]

        return text

    def parse(self, images: list, filter_imgtags: bool = True) -> list:
        vllm_inputs = [
            {
                "prompt": self.prompt,
                "multi_modal_data": {"image": image},
                "mm_processor_kwargs": {
                    "images_kwargs": {
                        "min_pixels": 448 * 448,
                        "max_pixels": 2880 * 2880,
                    }
                },
            }
            for image in images
        ]

        outputs = self.model.generate(vllm_inputs, self.sampling_params)

        markdowns = []
        for output in outputs:
            text = output.outputs[0].text.strip()
            if filter_imgtags:
                text = "\n\n".join(
                    block
                    for block in text.split("\n\n")
                    if not block.strip().startswith('<img src="images/bbox_')
                )
            markdowns.append(self._clean_truncated_repeats(text))

        return markdowns

## 4. Load the model

In [ ]:
parser = OvisOCR2Parser(MODEL_ID)

## 5. Markdown → standalone HTML
OvisOCR2 emits Markdown with raw HTML tables + LaTeX. We render it to HTML (raw HTML passes through), wrap it in the **same house CSS** used by the StructureV3 / VL 1.6 pages, and add MathJax so formulas display.

In [ ]:
import markdown as md_lib

HOUSE_CSS = (
    "body{font-family:Arial,Helvetica,sans-serif;max-width:900px;margin:24px auto;"
    "padding:0 16px;line-height:1.5}"
    "table{border-collapse:collapse;margin:12px 0}"
    "td,th{border:1px solid #888;padding:4px 8px;vertical-align:top}"
    "h1,h2,h3{margin:.6em 0 .3em}img{max-width:100%}"
)

MATHJAX = (
    "<script>window.MathJax={tex:{inlineMath:[['$','$'],['\\\\(','\\\\)']],"
    "displayMath:[['$$','$$'],['\\\\[','\\\\]']]}};</script>"
    "<script async src='https://cdn.jsdelivr.net/npm/mathjax@3/es5/tex-mml-chtml.js'></script>"
)


def md_to_html(markdown_text: str, title: str) -> str:
    body = md_lib.markdown(
        markdown_text,
        extensions=["tables", "fenced_code", "sane_lists"],
    )
    return (
        "<!doctype html>\n<html><head><meta charset='utf-8'>"
        f"<title>{title}</title><style>{HOUSE_CSS}</style>{MATHJAX}</head>\n"
        f"<body>\n{body}\n</body></html>\n"
    )

## 6. Run: one image at a time, capture what the comparison needs
We run images **individually** so per-image `latency` is real and comparable to the other models (which were timed the same way). For each image we save:

| artifact | why the local comparison needs it |
|---|---|
| `ovisocr2_md/<stem>.md` | raw model text → the plain-text token stream scored (word P/R/F1, char-sim) against VL |
| `ovisocr2_html/<stem>.html` | the deliverable, for eyeballing layout next to VL / StructureV3 |
| row in `ovisocr2_per_image.csv` | `image, category, latency, chars, n_tables` — table count feeds `table_recall`; latency/chars feed the resource table |

`n_tables` is counted exactly as in the main notebook: `md.lower().count("<table")`.

In [ ]:
import time, csv

if not image_paths:
    raise RuntimeError(
        f"No images found under IMAGE_DIR={IMAGE_DIR!r}. "
        "On Kaggle: attach the image dataset (Add Input) and point IMAGE_DIR at its "
        "mount under /kaggle/input; also check MAX_IMAGES is not 0."
    )

# fixed schema so the manifest header matches ocr_vl16_comparison.ipynb's per_image_metrics.csv
FIELDS = ["model", "key", "image", "category", "latency",
          "regions", "chars", "mean_conf", "n_tables"]

rows = []
t0 = time.time()

for idx, p in enumerate(image_paths, 1):
    img = Image.open(p).convert("RGB")
    t = time.time()
    md = parser.parse([img])[0]
    lat = time.time() - t

    stem = p.stem
    n_tables = md.lower().count("<table")
    (MD_DIR / f"{stem}.md").write_text(md, encoding="utf-8")
    (OUT_DIR / f"{stem}.html").write_text(md_to_html(md, p.name), encoding="utf-8")

    rows.append({
        "model": "OvisOCR2", "key": "ovisocr2", "image": p.name,
        "category": category_of(stem), "latency": round(lat, 3),
        "regions": "", "chars": len(md), "mean_conf": "", "n_tables": n_tables,
    })
    if idx % 10 == 0 or idx == len(image_paths):
        print(f"  {idx}/{len(image_paths)}  last={lat:.2f}s  tables={n_tables}")

elapsed = time.time() - t0
n = max(len(image_paths), 1)

with open(MANIFEST, "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=FIELDS)
    w.writeheader(); w.writerows(rows)

print(f"\nParsed {len(image_paths)} images in {elapsed:.1f}s  ({elapsed / n:.2f}s/img)")
print(f"HTML     -> {OUT_DIR}")
print(f"Markdown -> {MD_DIR}")
print(f"Manifest -> {MANIFEST}")

## 7. Preview a result inline

In [ ]:
from IPython.display import HTML, display

if image_paths:
    sample = OUT_DIR / f"{image_paths[0].stem}.html"
    print(sample)
    display(HTML(sample.read_text(encoding="utf-8")))

## 8. Zip the outputs, then download for the local comparison
Download `ovisocr2_outputs.zip` from the Kaggle **Output** tab. It contains the `.md` files and `ovisocr2_per_image.csv` — that is everything section 9 needs.

In [ ]:
import shutil, os

bundle = Path("/kaggle/working/ovisocr2_bundle")
bundle.mkdir(exist_ok=True)
shutil.copytree(MD_DIR, bundle / "ovisocr2_md", dirs_exist_ok=True)
shutil.copytree(OUT_DIR, bundle / "ovisocr2_html", dirs_exist_ok=True)
shutil.copy(MANIFEST, bundle / MANIFEST.name)
shutil.make_archive("/kaggle/working/ovisocr2_outputs", "zip", bundle)
print("Wrote /kaggle/working/ovisocr2_outputs.zip")

---
## 9. LOCAL comparison vs VL 1.6  ⟵ run this on your machine, not on Kaggle

This cell does **not** need a GPU or the model. Unzip `ovisocr2_outputs.zip` locally, point the two paths below at (a) the extracted `ovisocr2_md/` and (b) your existing `.../results/vl_1_6/` folder, and it will emit `quality_vs_vl_ovisocr2_per_image.csv` + a one-row summary.

It reuses the **identical** metric definitions from `ocr_vl16_comparison.ipynb` so the numbers are directly comparable to run1–run4:

- **VL 1.6 is the pseudo-reference** — its `<stem>.md` gives the reference text and its `<table` count.
- `normtext` → strip HTML/Markdown to a lowercase token stream.
- `word_prf` → multiset word precision / recall / F1 of OvisOCR2 vs VL.
- `char_sim` → difflib ratio on the normalized strings.
- `table_recall` → `min(model_tables, vl_tables) / vl_tables`.

The output CSV has the same columns as `quality_vs_vl_per_image.csv`, so you can concatenate it with the run4 file and drop OvisOCR2 straight into the comparison report.

In [ ]:
# ==== EDIT THESE TWO PATHS (local machine) ====
OVIS_MD_DIR = "ovisocr2_md"  # extracted from the Kaggle zip
VL_DIR      = "../experiments/100img_run4_sv3_ocrv6_lean/results/vl_1_6"
OUT_CSV     = "quality_vs_vl_ovisocr2_per_image.csv"
# ==============================================

import os, re, html, glob, difflib, csv
from collections import Counter

CATEGORIES = ("agenda", "form", "invoice", "report", "original")
def category_of(stem):
    for c in CATEGORIES:
        if stem.lower().startswith(c):
            return c
    return "other"

# --- metric functions, identical to ocr_vl16_comparison.ipynb ---
_TAG = re.compile(r"<[^>]+>")
_MD  = re.compile(r"[#*_`|>]+")
_WS  = re.compile(r"\s+")
def normtext(s):
    s = _TAG.sub(" ", s or ""); s = html.unescape(s); s = _MD.sub(" ", s)
    s = s.replace("-", " "); s = _WS.sub(" ", s).lower().strip()
    return s

def char_sim(a, b):
    return difflib.SequenceMatcher(None, a, b).ratio()

def word_prf(pred, ref):
    p, r = Counter(pred.split()), Counter(ref.split())
    inter = sum((p & r).values())
    P = inter / max(sum(p.values()), 1)
    R = inter / max(sum(r.values()), 1)
    F = 2 * P * R / max(P + R, 1e-9)
    return P, R, F

def n_tables(md):
    return md.lower().count("<table")

# --- load VL 1.6 reference (text + table count) per image stem ---
vl = {}
for f in glob.glob(os.path.join(VL_DIR, "*.md")):
    stem = os.path.splitext(os.path.basename(f))[0]
    raw = open(f, encoding="utf-8").read()
    vl[stem] = {"text": normtext(raw), "tables": n_tables(raw)}
print(f"Loaded {len(vl)} VL 1.6 reference pages")
if not vl:
    raise RuntimeError(f"No VL 1.6 .md files under VL_DIR={VL_DIR!r} — check the path.")

# --- score OvisOCR2 against VL ---
rows, missing = [], []
for f in sorted(glob.glob(os.path.join(OVIS_MD_DIR, "*.md"))):
    stem = os.path.splitext(os.path.basename(f))[0]
    if stem not in vl:
        missing.append(stem); continue
    raw = open(f, encoding="utf-8").read()
    pred, ref = normtext(raw), vl[stem]["text"]
    P, R, F = word_prf(pred, ref)
    cs = char_sim(pred, ref)
    vl_tab, m_tab = vl[stem]["tables"], n_tables(raw)
    tab_recall = (min(m_tab, vl_tab) / vl_tab) if vl_tab > 0 else ""
    rows.append({"model": "OvisOCR2", "key": "ovisocr2", "image": f"{stem}.jpg",
                 "category": category_of(stem), "word_P": P, "word_R": R, "word_F1": F,
                 "char_sim": cs, "vl_tables": vl_tab, "model_tables": m_tab,
                 "table_recall": tab_recall})

if missing:
    print(f"WARNING: {len(missing)} OvisOCR2 pages had no VL match (skipped): {missing[:5]}...")

if not rows:
    raise RuntimeError(
        f"No OvisOCR2 .md files in OVIS_MD_DIR={OVIS_MD_DIR!r} matched a VL page "
        f"({len(vl)} VL pages loaded). Point OVIS_MD_DIR at the extracted 'ovisocr2_md' "
        "folder and confirm the stems match the VL filenames (e.g. agenda_01)."
    )

FIELDS = ["model", "key", "image", "category", "word_P", "word_R", "word_F1",
          "char_sim", "vl_tables", "model_tables", "table_recall"]
with open(OUT_CSV, "w", newline="") as fh:
    w = csv.DictWriter(fh, fieldnames=FIELDS)
    w.writeheader(); w.writerows(rows)

# --- summary (means), mirroring quality_vs_vl_summary.csv ---
def mean(key, vals=rows):
    xs = [r[key] for r in vals if isinstance(r[key], (int, float))]
    return sum(xs) / len(xs) if xs else float("nan")

print(f"\nScored {len(rows)} pages -> {OUT_CSV}")
print("OvisOCR2 vs VL 1.6 (means):")
print(f"  word_P={mean('word_P'):.3f}  word_R={mean('word_R'):.3f}  "
      f"word_F1={mean('word_F1'):.3f}  char_sim={mean('char_sim'):.3f}  "
      f"table_recall={mean('table_recall'):.3f}")